Markdown:

#***Audio Data Preprocessing Pipeline~***

To prepare our audio data for machine learning models, we implement a standardized preprocessing workflow to clean and normalize all .wav tracks:

Resampling to a Common SR: Converts all audio tracks to a uniform sampling rate (e.g., 22050Hz or 16000Hz) to ensure data consistency across the model.

Ensuring it's Monophonic: Converts multi-channel (stereo) audio to a single channel (mono) by averaging channels, reducing computational complexity.

File Filtering: Automatically discards corrupted or very small/empty audio files that lack sufficient features for training.

Silence Trimming: Strips leading and trailing dead silence from clips using librosa.effects.trim to focus strictly on active signals.Removing

DC Offset: Centers the audio waveform around zero by subtracting the mean, preventing low-frequency distortion in subsequent feature extraction.

Amplitude Normalization: Scales the peak amplitude of all clips uniformly (e.g., to a range between $-1.0$ and $1.0$) so volume variations don't skew the model.

Windowing: Applies a window function (like a Hann window) during framing to minimize spectral leakage before applying Fourier Transforms.

Installing and Importing Libraries and mounting Google Drive

In [ ]:
!pip install librosa soundfile -q

import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import os
import glob
from google.colab import drive

drive.mount('/content/drive', force_remount=True)



Defing Paths and Parameters:

TARGET_SR : Change all the audios to a sampling rate of 16000 Hz

WINDOW_SEC : All the audios are splitted to 3 second clips

HOP_SEC : after each window we hop 1.5 second to start the next clip


In [ ]:
data_dir = '/content/drive/MyDrive/Underwater Audio Data/data'
output_dir = '/content/drive/MyDrive/Underwater Audio Data/processed'
classes = ['Vessels', 'Biological', 'Ambience']
classes = ['Vessels', 'Biological', 'Ambience']

TARGET_SR = 16000
WINDOW_SEC = 3.0
HOP_SEC = 1.5

This function helps in windowing by splitting given audio file into clips based on the WINDOE_SEC and HOP_SEC parameters.


In [ ]:
def window_audio(y, sr, window_sec=WINDOW_SEC, hop_sec=HOP_SEC):
    window_len = int(window_sec * sr)
    hop_len = int(hop_sec * sr)

    if len(y) <= window_len:
        return [np.pad(y, (0, window_len - len(y)))]

    chunks = []

    for start in range(0, len(y) - window_len + 1, hop_len):
        chunks.append(y[start:start + window_len])

    if (len(y) - window_len) % hop_len != 0:
        chunks.append(y[-window_len:])

    return chunks

This is the CORE Processing loop we use.

An empty list is initialised named RECORDS which stores the metadata of the processed files.

It loops through each class and each .wav file and performs a number of cleaning/pre-processing steps on the dataset.The steps are:


1. Resampling to a common SR
2. Ensuring its Monophonic
3. Corrupted and very small files are discarded

4. Silence Trimming
5. Removing DC offset
6. Normalising, all files have similar amplitude.

7. Windowing



In [ ]:
records = []

for label in classes:
    class_dir = os.path.join(data_dir, label)
    files = glob.glob(os.path.join(class_dir, '*.wav'))

    out_class_dir = os.path.join(output_dir, label)
    os.makedirs(out_class_dir, exist_ok=True)

    for f in files:
        try:
            y, sr = librosa.load(f, sr=TARGET_SR, mono=True)

            # Skip empty or small files
            if len(y) < int(0.2 * TARGET_SR):
                continue
            # Trim silence except for ambience
            if label != 'ambience':
                y, _ = librosa.effects.trim(y, top_db=25)

            # Remove DC offset
            y = y - np.mean(y)
            # Normalise
            peak = np.max(np.abs(y))
            if peak > 0:
                y = y / peak
            # Windowing into fixed time chunks
            chunks = window_audio(y, sr)

            base_name = os.path.splitext(os.path.basename(f))[0]

            for i, chunk in enumerate(chunks):
                out_path = os.path.join(
                    out_class_dir,
                    f"{base_name}_chunk{i}.wav"
                )

                sf.write(out_path, chunk, sr)

                records.append({
                    'file': out_path,
                    'label': label,
                    'source_file': f,
                    'duration': len(chunk) / sr
                })

        except Exception as e:
            print(f"Skipped {f}: {e}")

This means that the killerwhale(1).wav file is corrupted. As can be seen by running the following code.

In [ ]:
sf.info("/content/drive/MyDrive/Underwater Audio Data/data/Biological/killerwhale (1).wav")

Saves the metadata of the processed dataset and saves it as a .csv file named manifest.csv

**Also when we split the data, it is very important that clips(after windowing) from the same original recording stay in the same split. So we use the source_file coloumn in the manifest.csv**

In [ ]:
df_processed = pd.DataFrame(records)

df_processed.to_csv(
    os.path.join(output_dir, 'manifest.csv'),
    index=False
)

We can check the dataframe by the following operations.

In [ ]:
print(df_processed.head())

print(df_processed['label'].value_counts())

print(df_processed.shape)

This tells us the number of seconds of audio files in each class after cleaning the dataset.

                  
                

In [ ]:
print(df_processed.groupby("label")["duration"].sum())

Because of this imbalance it datasets of each class we have to use **Class weights**